# Bước 2 – Chuyển mô hình sang TFLite để chạy trên Edge Gateway (TV6)

Đầu vào là 2 mô hình đã huấn luyện ở bước 1 (`ml/models/*.joblib`). Notebook này:

1. Dựng lại từng mô hình bằng TensorFlow/Keras rồi chuyển sang **TFLite**:
   - **Phân loại sức khỏe đất:** lượng tử hóa **INT8**, tức trọng số và phép tính dạng số nguyên 8 bit. Cách này chạy nhanh và tiết kiệm năng lượng trên CPU ARM hoặc bộ tăng tốc NPU/Edge TPU.
   - **Phát hiện bất thường:** giữ **FP32**, vì điểm z-score trải rộng từ khoảng 1 đến vài trăm, lượng tử hóa INT8 sẽ làm mất độ chính xác ở ngưỡng.
2. Kiểm tra TFLite cho **cùng kết quả** với mô hình gốc trên tập test.
3. Chạy thử bằng [`edge/edge_runtime.py`](edge/edge_runtime.py), đúng đoạn code gateway sẽ dùng, và đo độ trễ từng mẫu.
4. Sinh `output/register_tflite.sql` để đăng ký version **1.2.0** (framework `TFLITE`) vào database.

**Môi trường:** cần TensorFlow ≥ 2.16. Trên Windows nên tạo venv ở đường dẫn ngắn, vì Python bản Microsoft Store vướng giới hạn 260 ký tự của đường dẫn. Xem `ml/README.md`.

In [1]:
import os, io, sys, json, time, hashlib, warnings, contextlib, logging
from pathlib import Path
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
tf.get_logger().setLevel('ERROR'); logging.getLogger('absl').setLevel(logging.ERROR)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

ML_DIR = Path.cwd() if (Path.cwd() / 'queries').exists() else Path.cwd() / 'ml'
DATA_DIR, MODEL_DIR, OUT_DIR = ML_DIR / 'data', ML_DIR / 'models', ML_DIR / 'output'
sys.path.insert(0, str(ML_DIR / 'edge'))
SEED = 42
tf.keras.utils.set_random_seed(SEED)
TZ = 'Asia/Ho_Chi_Minh'
THRESHOLDS = {'accuracy': 0.90, 'precision': 0.90, 'recall': 0.80, 'f1': 0.85, 'latency_ms': 2000}

def load(name):
    sql = (ML_DIR / 'queries' / f'{name}.sql').read_text(encoding='utf-8')
    if os.environ.get('DATABASE_URL'):
        import psycopg
        with psycopg.connect(os.environ['DATABASE_URL']) as conn:
            return pd.read_sql(sql, conn)
    return pd.read_csv(DATA_DIR / f'{name}.csv')

def scores(y_true, y_pred, average):
    return {'accuracy':  accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, average=average, zero_division=0),
            'recall':    recall_score(y_true, y_pred, average=average, zero_division=0),
            'f1':        f1_score(y_true, y_pred, average=average, zero_division=0)}

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

trained = json.loads((OUT_DIR / 'metrics.json').read_text(encoding='utf-8'))
cls_sk = joblib.load(MODEL_DIR / 'soil_health_cls.joblib')
cls_spec = json.loads((MODEL_DIR / 'soil_health_cls.json').read_text(encoding='utf-8'))
an_sk = joblib.load(MODEL_DIR / 'sensor_anomaly.joblib')
an_spec = json.loads((MODEL_DIR / 'sensor_anomaly.json').read_text(encoding='utf-8'))
print('TensorFlow', tf.__version__)
print('Mô hình bước 1:', {k: v['algorithm'] for k, v in trained['models'].items()}, '- version', trained['version'])

TensorFlow 2.21.0
Mô hình bước 1: {'SOIL-HEALTH-CLS': 'Logistic Regression', 'SENSOR-ANOMALY': 'Robust z-score (MAD)'} - version 1.1.0


## A. Phân loại sức khỏe đất → TFLite INT8

Mô hình bước 1 là Logistic Regression (chuẩn hóa + tuyến tính + softmax), nên dựng lại **chính xác** bằng Keras: lớp `Normalization` dùng mean/variance của bộ chuẩn hóa, lớp `Dense` dùng đúng hệ số đã học. Không huấn luyện lại, nên kết quả giữ nguyên.

Nếu sau này, với dữ liệu thật, bước 1 chọn mô hình dạng cây (Decision Tree / Random Forest), notebook sẽ tự **huấn luyện một mạng nơ-ron nhỏ** trên cùng tập train thay thế, vì TFLite không chạy trực tiếp mô hình cây.

In [2]:
FEATURES = cls_spec['features']
CLASSES = cls_spec['classes']
ds = load('dataset_records').dropna(subset=['label_code'])
part = {s: ds[ds['split'] == s] for s in ['TRAIN', 'VAL', 'TEST']}
Xtr = part['TRAIN'][FEATURES].to_numpy(np.float32)
Xte, yte = part['TEST'][FEATURES].to_numpy(np.float32), part['TEST']['label_code'].to_numpy()

def keras_from_sklearn(model):
    steps = dict(model.named_steps) if hasattr(model, 'named_steps') else {}
    scaler, lr = steps.get('standardscaler'), steps.get('logisticregression')
    if scaler is None or lr is None:
        return None
    coef, icpt = lr.coef_, lr.intercept_
    if coef.shape[0] == 1:                                   # nhị phân: logit lớp 0 = 0, lớp 1 = w·x + b
        W = np.vstack([np.zeros_like(coef), coef]).T
        b = np.array([0.0, icpt[0]])
    else:                                                    # đa lớp (multinomial)
        W, b = coef.T, icpt
    inp = tf.keras.Input(shape=(len(FEATURES),), name='features')
    norm = tf.keras.layers.Normalization(mean=scaler.mean_, variance=scaler.var_, name='scaler')
    dense = tf.keras.layers.Dense(len(CLASSES), activation='softmax', name='logreg')
    m = tf.keras.Model(inp, dense(norm(inp)))
    dense.set_weights([W.astype(np.float32), b.astype(np.float32)])
    return m

def keras_mlp():
    norm = tf.keras.layers.Normalization(name='scaler'); norm.adapt(Xtr)
    m = tf.keras.Sequential([tf.keras.Input(shape=(len(FEATURES),), name='features'), norm,
                             tf.keras.layers.Dense(16, activation='relu'), tf.keras.layers.Dense(16, activation='relu'),
                             tf.keras.layers.Dense(len(CLASSES), activation='softmax')])
    m.compile('adam', 'sparse_categorical_crossentropy', metrics=['accuracy'])
    ytr_i = pd.Categorical(part['TRAIN']['label_code'], categories=CLASSES).codes
    m.fit(Xtr, ytr_i, epochs=40, batch_size=64, verbose=0)
    return m

keras_cls = keras_from_sklearn(cls_sk)
conversion_method = 'chuyển chính xác hệ số Logistic Regression'
if keras_cls is None:
    keras_cls, conversion_method = keras_mlp(), 'huấn luyện MLP 16-16 thay thế'
print('Cách dựng:', conversion_method)

sk_pred = cls_sk.predict(Xte if not hasattr(cls_sk, 'feature_names_in_') else pd.DataFrame(Xte, columns=FEATURES))
keras_pred = np.array(CLASSES)[keras_cls.predict(Xte, verbose=0).argmax(1)]
print(f'Keras khớp scikit-learn: {np.mean(sk_pred == keras_pred):.4%} số mẫu test')

Cách dựng: chuyển chính xác hệ số Logistic Regression


Keras khớp scikit-learn: 100.0000% số mẫu test


In [3]:
def to_tflite(model, int8=False, rep=None):
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    if int8:
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.representative_dataset = lambda: ([x[None, :]] for x in rep)
        conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        # giữ input/output float32 để gateway gửi số đo thật, lượng tử hóa nằm bên trong model
    with contextlib.redirect_stdout(io.StringIO()):          # ẩn log "Saved artifact..." của TensorFlow
        return conv.convert()

rng = np.random.default_rng(SEED)
rep = Xtr[rng.choice(len(Xtr), 500, replace=False)]
tflite_fp32 = to_tflite(keras_cls)
tflite_int8 = to_tflite(keras_cls, int8=True, rep=rep)
(MODEL_DIR / 'soil_health_cls.tflite').write_bytes(tflite_int8)

def run_tflite(blob, X):
    it = tf.lite.Interpreter(model_content=blob); it.allocate_tensors()
    i, o = it.get_input_details()[0], it.get_output_details()[0]
    out = []
    for x in X:
        it.set_tensor(i['index'], x[None, :].astype(np.float32)); it.invoke()
        out.append(it.get_tensor(o['index'])[0])
    return np.array(out)

rows = []
for name, blob in [('Keras FP32 → TFLite FP32', tflite_fp32), ('TFLite INT8 (bản dùng trên gateway)', tflite_int8)]:
    pred = np.array(CLASSES)[run_tflite(blob, Xte).argmax(1)]
    rows.append({'phiên bản': name, 'kích thước (KB)': round(len(blob) / 1024, 2),
                 'khớp scikit-learn': round(float(np.mean(pred == sk_pred)), 4),
                 **{k: round(v, 4) for k, v in scores(yte, pred, 'macro').items()}})
rows.insert(0, {'phiên bản': 'scikit-learn (bước 1)', 'kích thước (KB)': trained['models']['SOIL-HEALTH-CLS']['size_kb'],
                'khớp scikit-learn': 1.0, **{k: round(v, 4) for k, v in scores(yte, sk_pred, 'macro').items()}})
cmp_cls = pd.DataFrame(rows).set_index('phiên bản')
display(cmp_cls)

,kích thước (KB),khớp scikit-learn,accuracy,precision,recall,f1
phiên bản,,,,,,
scikit-learn (bước 1),1.14,1.0,1.0,1.0,1.0,1.0
Keras FP32 → TFLite FP32,1.96,1.0,1.0,1.0,1.0,1.0
TFLite INT8 (bản dùng trên gateway),2.56,1.0,1.0,1.0,1.0,1.0


## B. Phát hiện bất thường → TFLite FP32

Mô hình z-score chỉ gồm một phép tính: `score = max_c |độ lệch_c| / MAD_c`. Phép tính này được đóng gói thành một mô hình TFLite rất nhỏ, để gateway nạp **cả hai mô hình theo cùng một cách** và cập nhật OTA giống nhau.

Phần tính độ lệch so với trung vị 8 bản tin trước nằm trong `edge_runtime.py`, vì nó cần nhớ lịch sử của từng cảm biến.

In [4]:
AN_FEATURES = an_spec['features']
mad = np.array([an_spec['mad'][f] for f in AN_FEATURES], dtype=np.float32)

class ZScore(tf.Module):
    @tf.function(input_signature=[tf.TensorSpec([1, len(AN_FEATURES)], tf.float32, name='deviation')])
    def __call__(self, dev):
        return tf.reduce_max(tf.abs(dev) / mad, axis=1, keepdims=True)

zmod = ZScore()
conv = tf.lite.TFLiteConverter.from_concrete_functions([zmod.__call__.get_concrete_function()], zmod)
with contextlib.redirect_stdout(io.StringIO()):
    tflite_an = conv.convert()
(MODEL_DIR / 'sensor_anomaly.tflite').write_bytes(tflite_an)
print(f'sensor_anomaly.tflite: {len(tflite_an)} byte, ngưỡng {an_spec["threshold"]:.4f}')

sensor_anomaly.tflite: 1340 byte, ngưỡng 31.6773


## C. Chạy thử bằng code của gateway (`edge_runtime.py`)

Cho toàn bộ bản tin cảm biến đi qua `EdgeModels.infer()` **lần lượt từng bản tin, theo thứ tự thời gian của từng cảm biến**, đúng như gateway nhận dữ liệu thật. Sau đó:

- So điểm bất thường với cách tính theo lô (pandas) của bước 1. Hai cách phải ra cùng một kết quả.
- Tính lại metrics trên **cùng tập test** như bước 1 (20% thời gian cuối).
- Đo độ trễ của từng lần suy luận.

Gateway chạy mô hình bất thường **trước**. Nếu bản tin bị đánh giá là bất thường, gateway **không** phân loại sức khỏe đất cho bản tin đó, vì số đo sai sẽ cho kết luận sai.

In [5]:
from edge_runtime import EdgeModels
edge = EdgeModels(MODEL_DIR)

readings = load('readings')
readings['measured_at'] = pd.to_datetime(readings['measured_at'], utc=True).dt.tz_convert(TZ)
r = readings.sort_values(['sensor_code', 'measured_at']).reset_index(drop=True)

results = []
for row in r.to_dict('records'):
    out = edge.infer(row)
    results.append({'anomaly_score': out['anomaly']['anomaly_score'], 'is_anomaly': out['anomaly']['is_anomaly'],
                    'predicted_label': out['health']['predicted_label'], 'confidence': out['health']['confidence'],
                    'lat_anomaly': out['anomaly']['latency_ms'], 'lat_health': out['health']['latency_ms']})
res = pd.concat([r, pd.DataFrame(results)], axis=1)
print(f'Đã suy luận {len(res)} bản tin')

Đã suy luận 12602 bản tin


In [6]:
# Cách tính theo lô của bước 1, để đối chiếu
g = r.groupby('sensor_code')
dev = pd.DataFrame({c: (r[c] - g[c].transform(lambda s: s.shift(1).rolling(8, min_periods=1).median())).abs().fillna(0)
                    for c in AN_FEATURES})
batch_score = (dev / pd.Series(an_spec['mad'])).max(axis=1)
batch_flag = batch_score >= an_spec['threshold']
print(f'Chênh lệch điểm lớn nhất (gateway so với lô): {np.max(np.abs(res.anomaly_score - batch_score)):.4f}')
print(f'Kết luận bất thường khớp nhau: {np.mean(res.is_anomaly == batch_flag):.4%}')

RANGE_FLAGS = {'PH_RANGE', 'PH_PLAUSIBLE', 'EC_RANGE', 'MOIST_RANGE', 'TEMP_RANGE', 'N_RANGE', 'P_RANGE', 'K_RANGE'}
res['anomaly'] = res['quality_flags'].fillna('').str.split('|').apply(lambda f: bool(set(f) & RANGE_FLAGS)).astype(int)
t2 = res['measured_at'].quantile(0.8)
te_an = res[res['measured_at'] > t2]
an_tflite = scores(te_an['anomaly'], te_an['is_anomaly'].astype(int), 'binary')

# Phân loại: chỉ đánh giá trên các bản tin thuộc tập test của dataset (cùng tập với bước 1)
test_ids = set(part['TEST']['observed_at'].astype(str) + '|' + part['TEST']['plot_code'])
res['key'] = res['measured_at'].dt.strftime('%Y-%m-%d %H:%M:%S%z').str[:-2] + '|' + res['plot_code']
key_label = dict(zip(part['TEST']['observed_at'].astype(str) + '|' + part['TEST']['plot_code'], part['TEST']['label_code']))
te_cls = res[res['key'].isin(test_ids)]
cls_tflite = scores(te_cls['key'].map(key_label), te_cls['predicted_label'], 'macro')
print(f'Bản tin tập test: phân loại {len(te_cls)}, bất thường {len(te_an)}')

Chênh lệch điểm lớn nhất (gateway so với lô): 0.0001
Kết luận bất thường khớp nhau: 100.0000%


Bản tin tập test: phân loại 2190, bất thường 2520


In [7]:
lat = {}
for name, col in [('SOIL-HEALTH-CLS', 'lat_health'), ('SENSOR-ANOMALY', 'lat_anomaly')]:
    v = res[col] if name == 'SENSOR-ANOMALY' else res.loc[res['predicted_label'].notna(), col]   # bỏ bản tin bị bỏ qua
    lat[name] = {'latency_ms_avg': float(v.mean()), 'latency_ms_p95': float(v.quantile(0.95)),
                 'latency_ms_max': float(v.max())}
display(pd.DataFrame(lat).T.round(3).rename(columns=lambda c: c.replace('latency_ms_', 'độ trễ (ms) ')))
print('Đo trên máy tính này (CPU x86). Raspberry Pi 5 thường chậm hơn vài lần, vẫn còn rất xa giới hạn 2000 ms.')

,độ trễ (ms) avg,độ trễ (ms) p95,độ trễ (ms) max
SOIL-HEALTH-CLS,0.056,0.089,0.371
SENSOR-ANOMALY,0.493,0.773,1.759


Đo trên máy tính này (CPU x86). Raspberry Pi 5 thường chậm hơn vài lần, vẫn còn rất xa giới hạn 2000 ms.


## D. Đối chiếu tiêu chí nghiệm thu (tập test, chạy bằng TFLite)

In [8]:
def acceptance(name, m, l):
    checks = [('Accuracy', m['accuracy'] * 100, '≥ 90%', m['accuracy'] >= THRESHOLDS['accuracy']),
              ('Precision', m['precision'] * 100, '≥ 90%', m['precision'] >= THRESHOLDS['precision']),
              ('Recall', m['recall'] * 100, '≥ 80%', m['recall'] >= THRESHOLDS['recall']),
              ('F1', m['f1'] * 100, '≥ 85%', m['f1'] >= THRESHOLDS['f1']),
              ('Suy luận p95 (ms)', l['latency_ms_p95'], '≤ 2000', l['latency_ms_p95'] <= THRESHOLDS['latency_ms'])]
    return pd.DataFrame([{'mô hình': name, 'chỉ số': c, 'TFLite': round(v, 3), 'yêu cầu': req, 'đạt': '✅' if ok else '❌'}
                         for c, v, req, ok in checks])

acc = pd.concat([acceptance('A. SOIL-HEALTH-CLS (INT8)', cls_tflite, lat['SOIL-HEALTH-CLS']),
                 acceptance('B. SENSOR-ANOMALY (FP32)', an_tflite, lat['SENSOR-ANOMALY'])])
display(acc.set_index(['mô hình', 'chỉ số']))

TFLite yêu cầu đạt
mô hình                   chỉ số                                
A. SOIL-HEALTH-CLS (INT8) Accuracy           100.000   ≥ 90%   ✅
                          Precision          100.000   ≥ 90%   ✅
                          Recall             100.000   ≥ 80%   ✅
                          F1                 100.000   ≥ 85%   ✅
                          Suy luận p95 (ms)    0.089  ≤ 2000   ✅
B. SENSOR-ANOMALY (FP32)  Accuracy           100.000   ≥ 90%   ✅
                          Precision          100.000   ≥ 90%   ✅
                          Recall             100.000   ≥ 80%   ✅
                          F1                 100.000   ≥ 85%   ✅
                          Suy luận p95 (ms)    0.773  ≤ 2000   ✅

## E. Lưu và sinh SQL đăng ký version 1.2.0

- `models/soil_health_cls.tflite`, `models/sensor_anomaly.tflite`: file sẽ được đóng gói và gửi xuống gateway qua OTA ở bước 4.
- `output/metrics_tflite.json`: metrics và độ trễ.
- `output/register_tflite.sql`: đăng ký version `1.2.0` (framework `TFLITE`, kèm SHA-256 của file) và metrics vào database. Chạy lại nhiều lần vẫn an toàn.

In [9]:
VERSION = '1.2.0'
built_at = pd.Timestamp.now(tz=TZ).isoformat(timespec='seconds')
q = lambda s: "'" + str(s).replace("'", "''") + "'"

items = {
    'SOIL-HEALTH-CLS': {'file': 'soil_health_cls.tflite', 'quant': 'INT8', 'spec': cls_spec, 'test': cls_tflite,
                        'lat': lat['SOIL-HEALTH-CLS'], 'algorithm': trained['models']['SOIL-HEALTH-CLS']['algorithm']},
    'SENSOR-ANOMALY':  {'file': 'sensor_anomaly.tflite', 'quant': 'FP32', 'spec': an_spec, 'test': an_tflite,
                        'lat': lat['SENSOR-ANOMALY'], 'algorithm': trained['models']['SENSOR-ANOMALY']['algorithm']},
}
sql = [f'-- Sinh tự động bởi ml/02_convert_tflite.ipynb lúc {built_at}. Chạy lại an toàn (upsert).', 'BEGIN;']
summary = {'version': VERSION, 'built_at': built_at, 'tensorflow': tf.__version__, 'source_version': trained['version'], 'models': {}}
for code_, it in items.items():
    path = MODEL_DIR / it['file']
    digest, size_kb = sha256(path), round(path.stat().st_size / 1024, 2)
    summary['models'][code_] = {'artifact': f'ml/models/{it["file"]}', 'sha256': digest, 'size_kb': size_kb,
                                'quantization': it['quant'], 'test': it['test'], **it['lat']}
    hyper = {'algorithm': it['algorithm'], 'converted_from': trained['version'], 'tensorflow': tf.__version__}
    sql.append(f'''
INSERT INTO ml.model_versions (model_id, version, dataset_version_id, framework, quantization, artifact_uri,
                               artifact_sha256, size_kb, input_schema, hyperparameters, target_hardware,
                               min_firmware_version, status, trained_at)
SELECT m.model_id, {q(VERSION)},
       (SELECT dataset_version_id FROM dataset.dataset_versions WHERE status = 'FROZEN' ORDER BY frozen_at DESC LIMIT 1),
       'TFLITE', {q(it['quant'])}, {q('ml/models/' + it['file'])}, {q(digest)}, {size_kb},
       {q(json.dumps(it['spec'], ensure_ascii=False))}::jsonb, {q(json.dumps(hyper))}::jsonb,
       'Raspberry Pi 5', '1.0.0', 'TRAINED', {q(built_at)}
FROM ml.models m WHERE m.model_code = {q(code_)}
ON CONFLICT (model_id, version) DO UPDATE
   SET artifact_sha256 = EXCLUDED.artifact_sha256, size_kb = EXCLUDED.size_kb, input_schema = EXCLUDED.input_schema,
       hyperparameters = EXCLUDED.hyperparameters, trained_at = EXCLUDED.trained_at;''')
    vals = {**it['test'], 'latency_ms_avg': it['lat']['latency_ms_avg'], 'latency_ms_p95': it['lat']['latency_ms_p95'],
            'flash_kb': size_kb}
    for k, v in vals.items():
        sql.append(f'''INSERT INTO ml.model_metrics (model_version_id, dataset_version_id, split, environment, metric_name, metric_value)
SELECT mv.model_version_id, mv.dataset_version_id, 'TEST', 'OFFLINE', {q(k.upper())}, {round(float(v), 6)}
FROM ml.model_versions mv JOIN ml.models m USING (model_id)
WHERE m.model_code = {q(code_)} AND mv.version = {q(VERSION)}
ON CONFLICT (model_version_id, split, environment, metric_name) DO UPDATE SET metric_value = EXCLUDED.metric_value, evaluated_at = now();''')
sql.append('COMMIT;')
(OUT_DIR / 'register_tflite.sql').write_text('\n'.join(sql) + '\n', encoding='utf-8')
(OUT_DIR / 'metrics_tflite.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
for c, m in summary['models'].items():
    print(f"{c:16} {m['quantization']:5} {m['size_kb']:>6} KB  sha256 {m['sha256'][:16]}…")
print('Đã ghi', OUT_DIR / 'register_tflite.sql')

SOIL-HEALTH-CLS  INT8    2.56 KB  sha256 9458d124b5627ef8…
SENSOR-ANOMALY   FP32    1.31 KB  sha256 5b1942221a6ca252…
Đã ghi C:\Users\admin\OneDrive\Documents\phát triển ứng dụng nâng cao\edge_aiot_biochar_db\ml\output\register_tflite.sql


## Kết luận

- Cả hai mô hình đã chạy được bằng TFLite qua đúng đoạn code gateway sẽ dùng (`edge_runtime.py`), và cho **cùng kết quả** với mô hình gốc trên tập test.
- **Phân loại sức khỏe đất:** bản INT8 có kết quả giống hệt bản gốc. Với mô hình cực nhỏ như thế này (14 tham số), file INT8 không nhỏ hơn bản FP32, vì phần thông tin lượng tử hóa chiếm chỗ. Lợi ích của INT8 nằm ở phép tính số nguyên chạy được trên NPU/Edge TPU, và sẽ rõ hơn khi dùng mô hình lớn hơn với dữ liệu thật.
- **Phát hiện bất thường:** cách tính từng bản tin một trên gateway cho cùng điểm số với cách tính theo lô ở bước 1. Điều này xác nhận gateway tính độ lệch đúng như lúc huấn luyện.
- Độ trễ đo trên máy tính này **chưa phải** độ trễ trên Raspberry Pi. Khi TV2 có bộ giả lập gateway hoặc thiết bị thật, chạy lại `edge_runtime.py` trên đó và ghi metric với `environment = 'EDGE'`.
- **Bước tiếp theo:** bước 3 là API model registry; bước 4 là đóng gói 2 file `.tflite` thành gói OTA (dùng SHA-256 đã tính ở đây) và triển khai xuống gateway.